Here is the code where you put in H, M, G, model and measurement

In [49]:
#| default_exp prediction_module

In [37]:
#| export
import pickle
import pandas as pd 
from anthropmass.data_module import *
from anthropmass.anthro_module import *
from anthropmass.bambi_module import *

Normalize is in data module but i dont know how to import

This function is normalizing the persons weight and height

In [38]:
#| export
def normalize_person(weight, height, gender):
    person=pd.DataFrame({'weightkg': [weight], 'stature': [height], 'Gender': [gender]})
    normalize(person, 'weightkg')
    normalize(person, 'stature')
    return person

This functions gets the pickled model

In [39]:
#| export
def get_pickled_model(kindofmodel:str, measurement:str):
    filepath = f'../output/anthro_models/{kindofmodel}/pickle_{measurement}_{kindofmodel}'
    with open(filepath,'rb') as file:
        model=pickle.load(file)
    return model

Predict_from_model uses the pickled model and the normalized person to predict the measurement

In [40]:
#| export
def predict_from_model(kindofmodel:str, measurement:str, w, h, g):
    pickledmodel = get_pickled_model(kindofmodel, measurement)
    person = normalize_person(w, h, g)
    if kindofmodel=='xgboost':
        return pickledmodel.predict(person)
    elif kindofmodel=='bambi':
        model = model_bmb(measurement)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    elif kindofmodel=='bambi_ml_gc':
        train=pd.read_csv('../data/processed/ANSURIInormalizedtrain.csv')
        model = component_model(measurement,train)
        person['Component']='Army Reserve' #'Regular Army'#'Army National Guard'#'Army Reserve'
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    else:
        return 'wrong model name'

In [41]:
#| export
def add_to_df(df, measurement, pred):
    df[measurement]=pred
    return df

I dont know if we need to make it to csv?

In [42]:
#| export
def make_csv(data, name):
    data.to_csv(f'{name}.csv')

Here are all predictions made, the measurements are in a list

In [43]:
#| export
def make_predictions(kindofmodel:str, measurements:list, w, h, g):
    output=pd.DataFrame()
    for m in measurements:
        pred = predict_from_model(kindofmodel, m, w, h, g)
        add_to_df(output, m, pred)
    return output

In [57]:
make_predictions('bambi_ml_gc',['abdominalextensiondepthsitting','acromialheight','neckcircumference'], 80, 1700, 1)

,abdominalextensiondepthsitting,acromialheight,neckcircumference
0,334.035033,-88.869652,397.015988


In [56]:
make_predictions('bambi',['abdominalextensiondepthsitting','acromialheight'], 80, 1700, 1)

,abdominalextensiondepthsitting,acromialheight
0,250.066345,1390.5659


In [54]:
make_predictions('xgboost',['abdominalextensiondepthsitting','acromialheight','neckcircumference'], 80, 1700, 1)

,abdominalextensiondepthsitting,acromialheight,neckcircumference
0,250.98558,1392.302124,395.754181


In [ ]:
train=pd.read_csv('../data/processed/ANSURIInormalizedtrain.csv')
y_train=train.iloc[:,1:94].drop('weightkg',axis=1).drop('stature',axis=1)
variables = y_train.columns[:]

df = make_predictions('xgboost',variables, 80, 1800, 1)
df

In [50]:
import nbdev; nbdev.nbdev_export()